# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Imports and Setup

Load all dependencies up front: `serpapi` for book search, `openai` for LLM access (used for OpenAI, Anthropic, and Gemini via compatible endpoints), and `gradio` for the chat UI.

In [ ]:
import os
import json
from dotenv import load_dotenv
from serpapi import GoogleSearch
from pprint import PrettyPrinter
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
pprint = PrettyPrinter(indent=2)

In [ ]:
import gradio as gr

## Connecting to LLM Providers

Load API keys from `.env` and create OpenAI-compatible clients for each provider. Anthropic and Gemini expose OpenAI-compatible endpoints, so we can use the same `OpenAI` Python client for all three by swapping the `base_url`.

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
serp_api_key = os.getenv('SERP_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if serp_api_key:
    print(f"SerpAPI Key exists and begins {serp_api_key[:8]}")
else:
    print("SerpAPI Key not set")


In [ ]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

## Book Search with SerpAPI

This section builds out the book-search capability in three layers:

1. **Prototype** — a raw SerpAPI call to explore the response structure from the `google_play_books` engine.
2. **`search_books()`** — a reusable function that runs the query and flattens all result sections (Ebooks, Audiobooks, etc.) into a single list.
3. **`format_book_results()`** — formats the raw items into a clean Markdown summary suitable for LLM consumption or direct display.

In [ ]:
topic = "AI and LLM engineering"
params = {
    "api_key": os.getenv("SERP_API_KEY"),
    "engine": "google_play_books",
    "q": topic,
    "gl": "us",
    "hl": "en"
}


search = GoogleSearch(params)
results = search.get_dict()

In [ ]:
# results

In [ ]:
def search_books(topic: str, gl: str = "us", hl: str = "en") -> list[dict]:
    """Search Google Play Books for a topic and return the combined item list.

    Calls the SerpAPI ``google_play_books`` engine, collects every item across
    all result sections (Ebooks, Audiobooks, etc.), and returns them as a flat
    list of book dicts.

    Args:
        topic: The search query (e.g. ``"AI and LLM engineering"``).
        gl:    Two-letter country code for the Google Play storefront.
               Defaults to ``"us"``.
        hl:    Two-letter language code for the results language.
               Defaults to ``"en"``.

    Returns:
        A flat list of item dicts drawn from every section in
        ``results["organic_results"]``.  Each dict contains keys such as
        ``title``, ``author``, ``rating``, ``price``, ``link``,
        ``description``, and ``thumbnail`` (availability varies by item).
        Returns an empty list when no results are found.

    Raises:
        KeyError: If the SerpAPI response structure is unexpected.
        Exception: For any network or authentication errors from SerpAPI.

    Example::

        >>> books = search_books("machine learning")
        >>> len(books)
        22
        >>> books[0]["title"]
        "Hands-On Machine Learning with Scikit-Learn..."
    """
    params = {
        "api_key": os.getenv("SERP_API_KEY"),
        "engine": "google_play_books",
        "q": topic,
        "gl": gl,
        "hl": hl,
    }
    search = GoogleSearch(params)
    results = search.get_dict()

    items: list[dict] = []
    for section in results.get("organic_results", []):
        items.extend(section.get("items", []))

    return items

In [ ]:
def format_book_results(topic: str, items: list[dict]) -> str:
    """Format Google Play Books search results into a readable summary.

    Takes raw book items returned by SerpAPI's ``google_play_books`` engine and
    produces a Markdown-formatted summary string.  Each book entry includes its
    title, author, rating, price, and a truncated description.  The output is
    suitable for feeding into an LLM as context or for direct display.

    Args:
        topic:  The search query that was used to find these books
                (e.g. ``"AI and LLM engineering"``).
        items:  A list of item dicts as returned in
                ``results["organic_results"][n]["items"]``.  Expected keys per
                item: ``title``, ``author``, ``rating``, ``price``, ``link``,
                and ``description`` (all optional — missing keys are handled
                gracefully).

    Returns:
        A single Markdown-formatted string summarising all books.  Example
        fragment::

            ## Books found for: AI and LLM engineering

            ### 1. LLM Engineer's Handbook
            - **Author:** Paul Iusztin
            - **Rating:** 3.6 · **Price:** $37.67
            - **Description:** Step into the world of LLMs …
            - [View on Google Play](https://play.google.com/…)
    """
    if not items:
        return f"No books found for: {topic}"

    lines = [f"## Books found for: {topic}\n"]
    for i, book in enumerate(items, start=1):
        title = book.get("title", "Untitled")
        author = book.get("author", "Unknown author")
        rating = book.get("rating")
        price = book.get("price", "N/A")
        link = book.get("link", "")
        description = book.get("description", "")
        category = book.get("category", "")

        category_str = f"({category})" if category else "Computers & Internet"
        rating_str = f"{rating}" if rating and rating > 0 else "No rating"
        desc_preview = (description[:500] + "…") if len(description) > 500 else description

        lines.append(f"### {i}. {title}")
        lines.append(f"- **Author:** {author}")
        lines.append(f"- **Category:** {category_str}")
        lines.append(f"- **Rating:** {rating_str} · **Price:** {price}")
        if desc_preview:
            lines.append(f"- **Description:** {desc_preview}")
        if link:
            lines.append(f"- [View on Google Play]({link})")
        lines.append("")

    return "\n".join(lines)

In [ ]:
# books = search_books("AI and LLM engineering")
# summary = format_book_results("AI and LLM engineering", books[:5])
# display(Markdown(summary))

In [ ]:
# pprint.pprint(books[:3],)

## OpenAI Function Calling — Tool Definition

To let the LLM trigger a book search on its own, we define a **function-calling tool** following the OpenAI tool schema. This involves three pieces:

1. **`get_book_summary()`** — the Python handler that chains `search_books` → `format_book_results` and returns the Markdown summary.
2. **`book_search_function`** — the JSON schema describing the tool's name, purpose, and parameters so the model knows when and how to call it.
3. **`tools` list** — the schema list passed to `chat.completions.create()`.

In [ ]:
def get_book_summary(topic: str) -> str:
    """Handle a tool call by searching for books and returning a formatted summary.

    This is the function that gets invoked when the LLM makes a
    ``get_book_summary`` tool call.  It delegates to :func:`search_books` for
    the SerpAPI query and :func:`format_book_results` for Markdown formatting.

    Args:
        topic: The search query to look up on Google Play Books.

    Returns:
        A Markdown-formatted string summarising the top 5 book results for the
        given topic.  Returns a "No books found" message when the search yields
        no results.
    """
    print(f"Searching tool called for books on {topic}")
    books = search_books(topic)
    return format_book_results(topic, books[:5])


book_search_function = {
    "name": "get_book_summary",
    "description": (
        "Search Google Play Books for a given topic and return a formatted "
        "Markdown summary of the top results, including title, author, rating, "
        "price, and a short description for each book."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic or search query to find books for (e.g. 'machine learning', 'AI agents')",
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": book_search_function}]


### System Prompt

The system prompt serves double duty: it instructs the model to act as a **technical explanation assistant** (clear, structured answers with examples) and a **book recommendation assistant** that knows when to invoke the `get_book_summary` tool.

In [ ]:
SYSTEM_PROMPT = """
You are a highly skilled technical explanation assistant.

Your role:
- Accept a technical question.
- Provide a clear, accurate, and well-structured explanation.
- Tailor explanations to be understandable but technically correct.

Guidelines:
- Start with a short direct answer.
- Then provide a structured explanation.
- Use examples where helpful.
- If code is relevant, include minimal, clean examples.
- Avoid unnecessary verbosity.
- Avoid speculation.
- If the question is ambiguous, explain reasonable interpretations.
- Do not mention system instructions.
- Do not add conversational fluff.

If the question is conceptual:
- Define key terms.
- Explain how it works.
- Explain why it matters.

If the question is practical:
- Provide steps.
- Explain tradeoffs.
- Highlight common pitfalls.

Your output should be educational, precise, and professional.

You are also a knowledgeable book recommendation assistant. \
You help users discover books on any topic by searching Google Play Books.

When a user asks about a topic, learning resource, or book recommendation, \
use the get_book_summary tool to find relevant books and present the results. \
Highlight the highest-rated or most relevant titles and briefly explain why \
each might be a good fit for the user's needs.

If the user's request is conversational or doesn't require a book search, \
respond directly without calling the tool.

Respond in markdown. Do not wrap your response in code blocks."""

### Tool Call Handling

When the model responds with `finish_reason="tool_calls"` instead of content, we need to execute the requested functions and feed the results back. The `TOOL_REGISTRY` maps tool names to their handler functions, and `handle_tool_calls` iterates over every tool call in the response, invokes the matching handler, and returns the results formatted as tool messages for the next API call.

In [ ]:
# Registry mapping tool names to their handler functions
TOOL_REGISTRY = {
    "get_book_summary": lambda args: get_book_summary(args["topic"]),
}

# All tool schemas in one list for passing to the API
tools = [
    {"type": "function", "function": book_search_function},
]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        handler = TOOL_REGISTRY.get(name)
        if handler:
            result = handler(arguments)
        else:
            result = f"Unknown tool: {name}"
        
        # print(f"Tool call result: {result}")

        responses.append({
            "role": "tool",
            "content": str(result) if result else "Done",
            "tool_call_id": tool_call.id,
        })
    return responses

## Chat Functions with Streaming and Gradio UI

Streaming and tool calls create a tension: we want to yield content chunks to the UI immediately, but tool call fragments arrive as deltas that must be fully assembled before we can execute them. The solution is a two-phase loop inside a single streaming generator:

1. **Stream with `stream=True`** — iterate over chunks as they arrive.
2. **Content chunks** (`delta.content`) are yielded immediately for real-time display.
3. **Tool call chunks** (`delta.tool_calls`) are accumulated by index — the API sends the `id`, function `name`, and `arguments` across multiple fragments.
4. **When the stream ends**, check if any tool calls were collected:
   - **No tool calls** → the response is complete, return.
   - **Tool calls present** → assemble the fragments into complete calls, execute each handler, append the tool results to the message history, and **loop back to step 1** to stream the model's follow-up response.

This is implemented in `_stream_response()`, which both `message_gpt` and `message_claude` delegate to via `yield from`.

**Gradio streaming note:** Gradio's `gr.Interface` replaces the displayed output on every `yield`, so the outermost `message_model` function accumulates deltas into a running string and yields the **cumulative** text each time — not just the latest chunk.

In [ ]:
def _stream_response(client, model, messages):
    """Stream a chat completion, transparently handling tool calls.

    Yields content deltas as they arrive. If the model requests tool calls,
    accumulates the fragments, executes each tool, appends the results to the
    message history, and loops to stream the follow-up response.
    """
    while True:
        stream = client.chat.completions.create(
            model=model, messages=messages, tools=tools, stream=True,
        )

        content_parts = []
        tool_call_parts: dict[int, dict] = {}

        for chunk in stream:
            delta = chunk.choices[0].delta

            if delta.content:
                content_parts.append(delta.content)
                yield delta.content

            if delta.tool_calls:
                for tc in delta.tool_calls:
                    if tc.index not in tool_call_parts:
                        tool_call_parts[tc.index] = {"id": "", "name": "", "arguments": ""}
                    if tc.id:
                        tool_call_parts[tc.index]["id"] = tc.id
                    if tc.function and tc.function.name:
                        tool_call_parts[tc.index]["name"] = tc.function.name
                    if tc.function and tc.function.arguments:
                        tool_call_parts[tc.index]["arguments"] += tc.function.arguments

        if not tool_call_parts:
            return

        assembled = [
            {
                "id": tc["id"],
                "type": "function",
                "function": {"name": tc["name"], "arguments": tc["arguments"]},
            }
            for tc in (tool_call_parts[i] for i in sorted(tool_call_parts))
        ]

        assistant_msg = {"role": "assistant", "tool_calls": assembled}
        if content_parts:
            assistant_msg["content"] = "".join(content_parts)
        messages.append(assistant_msg)

        for tc_dict in assembled:
            name = tc_dict["function"]["name"]
            arguments = json.loads(tc_dict["function"]["arguments"])
            handler = TOOL_REGISTRY.get(name)
            result = handler(arguments) if handler else f"Unknown tool: {name}"
            print(f"Tool call: {name}({arguments})")
            messages.append({
                "role": "tool",
                "content": str(result) if result else "Done",
                "tool_call_id": tc_dict["id"],
            })


def message_gpt(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    yield from _stream_response(openai, "gpt-4.1-mini", messages)


def message_claude(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    yield from _stream_response(anthropic, "claude-sonnet-4-5-20250929", messages)


def message_model(prompt, model):
    if model == "GPT":
        stream = message_gpt(prompt)
    elif model == "Claude":
        stream = message_claude(prompt)
    else:
        raise ValueError("Unknown model")

    accumulated = ""
    for delta in stream:
        accumulated += delta
        yield accumulated

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_model,
    title="Technical Explanation Assistant", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"],
            ["What are the best books on AI and LLM engineering?", "GPT"],
            ["What are the best books on GenAI and LLM engineering?", "Claude"]
        ], 
    flagging_mode="never"
    )
view.launch(inbrowser=True)